## A example showing how to use prompt chaining pattern

This chain contains 3 elements
* Document outliner
* Outline verifier
* Document finalizer

In [1]:
from pydantic import BaseModel,Field
from typing import List,Optional
from dotenv import load_dotenv
from openai import OpenAI

import logging

# Load all environment variables from .env file
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger(__name__)

In [2]:
# Define the input schema for the prompt chaining

class DocumentOutline(BaseModel):
    """A class to outline the structure of a document based on its content."""
    topic: str = Field(description="The main topic of the document.")
    sections: List[str] = Field(description="A list of section titles that outline the document structure.")

class OutlineValidator(BaseModel):
    """A class to validate the generated outline against the original document content."""
    is_valid: bool = Field(description="Indicates whether the outline is valid based on the document content.")
    validation_reason: str = Field(description="A reason explaining why the outline is valid or invalid.")
    validation_score: float = Field(description="A score between 0 and 1 indicating the confidence level of the validation.")

class FinalDocument(BaseModel):
    """A class to represent the final structured document after prompt chaining."""
    title: str = Field(description="The title of the final document.")
    content: str = Field(description="The main content of the final document, structured according to the outline.")

In [20]:
# Define LLM calls for each step in the prompt chaining process

client = OpenAI()
llm_model = "gpt-4o"

def generate_document_outline(topic: str) -> DocumentOutline:
    """First LLM call: generate a structured outline from a topic."""
    logger.info(f"Starting outline generation for topic: '{topic}'")

    llm_request = client.beta.chat.completions.parse(
        model=llm_model,
        messages=[
            {
                "role": "system",
                "content": """You are an expert content strategist.
                       Create a logical and comprehensive outline for a document on the given topic.
                       The outline should include an introduction, several body sections, and a conclusion.""",
            },
            {
                "role": "user",
                "content": topic,
            }
        ],
        response_format=DocumentOutline
    )
    result = llm_request.choices[0].message.parsed
    logger.info("Outline generation completed.")
    return result

def validate_document_outline(document_outline: DocumentOutline) -> OutlineValidator:
    """Second LLM call to validate the quality of the generated outline."""
    logger.info("Starting outline validation.")

    llm_request = client.beta.chat.completions.parse(
        model=llm_model,
        messages=[
            {
                "role": "system",
                "content": """You are a critical quality assurance editor. Your primary goal is to REJECT "
                    "low-quality or vague outlines. An outline is considered invalid if the "
                    "original topic is too vague, ambiguous, or lacks a clear focus (e.g., 'stuff', "
                    "'things', 'an article'). Be strict. If the topic is bad, the outline is bad. "
                    "Provide a brief reason for your decision."""
            },
            {
                "role": "user",
                "content": "Here is the document outline to validate:\n\n" + str(document_outline.model_dump()),
            }
        ],
        response_format=OutlineValidator
    )
    result = llm_request.choices[0].message.parsed
    logger.info("Outline validation completed.")
    return result

def document_finaliser(document_outline: DocumentOutline) -> FinalDocument:
    """Third LLM call: expand the validated outline into a full document."""
    logger.info("Generating final document from outline.")

    llm_request = client.beta.chat.completions.parse(
        model=llm_model,
        messages=[
            {
                "role": "system",
                "content": """You are a skilled author.
                    Write a comprehensive, well-structured document based on the provided outline.
                    Include an engaging title, clear section headings, and a concise conclusion.""",
            },
            {
                "role": "user",
                "content": "Here is the validated document outline to expand:\n\n" + str(document_outline.model_dump()),
            }
        ],
        response_format=FinalDocument
    )
    result = llm_request.choices[0].message.parsed
    logger.info("Final document generation completed.")
    return result



In [21]:
def chain_prompts(topic: str) -> Optional[FinalDocument]:
    """Main function implementing the prompt chain with a validation gate."""
    logger.info(f"Starting document creation process for topic: '{topic}'")

    # step 1: generate an outline
    outline = generate_document_outline(topic)

    # step 2: validate the outline
    validation_result = validate_document_outline(outline)

    if not validation_result.is_valid and validation_result.validation_score < 0.8:
        logger.warning(f"Outline validation failed: {validation_result.validation_reason}")
        return None  # Stop the chain if the outline is invalid
    else:
        final_document = document_finaliser(outline)
        logger.info("Document creation process completed successfully.")
        return final_document

In [22]:
input_topic = "TPUs and their impact on machine learning workloads"

final_doc = chain_prompts(input_topic)
if final_doc:
    logger.info(f"Final Document Title: {final_doc.title}")
    logger.info(f"Final Document Content:\n{final_doc.content}")
    print(f"Final Document Title: {final_doc.title}")
    print(f"Final Document Content:\n{final_doc.content}")
else:
    logger.error("Document creation failed due to invalid outline.")

2026-05-20 12:33:51 - INFO - Starting document creation process for topic: 'TPUs and their impact on machine learning workloads'
2026-05-20 12:33:51 - INFO - Starting outline generation for topic: 'TPUs and their impact on machine learning workloads'
2026-05-20 12:33:53 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-20 12:33:53 - INFO - Outline generation completed.
2026-05-20 12:33:53 - INFO - Starting outline validation.
2026-05-20 12:33:54 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-20 12:33:54 - INFO - Outline validation completed.
2026-05-20 12:33:54 - INFO - Generating final document from outline.
2026-05-20 12:34:11 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-20 12:34:11 - INFO - Final document generation completed.
2026-05-20 12:34:11 - INFO - Document creation process completed successfully.
2026-05-20 12:34:11 - INFO - Fin

Final Document Title: TPUs and Their Impact on Machine Learning Workloads
Final Document Content:


### Introduction to TPUs

In recent years, the landscape of machine learning and artificial intelligence has been significantly transformed by the advent of specialized hardware accelerators. Among these, Tensor Processing Units (TPUs) have emerged as a pivotal technology, promising unparalleled efficiency and speed in handling complex computations. This document explores the intricacies of TPUs, assessing their profound impact on machine learning workloads and uncovering their potential to revolutionize data processing in the AI domain.

### Overview of Machine Learning Workloads

Machine learning workloads are diverse and computationally intensive, requiring substantial processing power to train and deploy models effectively. These tasks vary from simple linear regressions to sophisticated deep learning networks capable of processing vast datasets. As data grows in complexity and size,